# 🧠 Sentiment Analyser — Training & Evaluation Demo
### Final Year Major Project

This notebook demonstrates the **complete ML pipeline** from dataset loading to model evaluation.

---

**Contents:**
1. Environment & imports
2. Dataset exploration (Amazon Reviews + SST-2)
3. Text preprocessing demonstration
4. Train/Val/Test split analysis
5. Model architecture summary
6. Training loop (short demo)
7. Evaluation metrics
8. Visualizations
9. Live prediction demo
10. BERT vs VADER comparison
11. Sarcasm detection demo
12. Export metrics

## 1. Environment & Imports

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

from config import Config
cfg = Config()

print('✅ Imports successful')
print(f'   Sentiment labels : {cfg.SENTIMENT_LABELS}')
print(f'   BERT model       : {cfg.BERT_MODEL_NAME}')
print(f'   Max length       : {cfg.MAX_LENGTH} tokens')

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version : {torch.__version__}')
print(f'Device          : {device}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU — training will be slow on CPU. Consider using --eval-only flag.')

## 2. Dataset Exploration

**Primary:** `amazon_polarity` — 3.6M Amazon product reviews  
**Neutral:** `sst2` — Stanford Sentiment Treebank (relabeled as Neutral)  

We load a sample for exploration (10,000 samples is fast for the notebook).

In [ ]:
from utils.data_loader import load_amazon_reviews, split_dataset, compute_class_weights

DEMO_SAMPLES = 10000   # ← Use 60000 for full training

print(f'Loading {DEMO_SAMPLES:,} samples from Amazon Reviews...')
df = load_amazon_reviews(max_samples=DEMO_SAMPLES)

print(f'\nDataset shape  : {df.shape}')
print(f'Columns        : {df.columns.tolist()}')
print(f'\nLabel distribution:')
for label, name in enumerate(['Negative', 'Neutral', 'Positive']):
    cnt = (df['label'] == label).sum()
    pct = cnt / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'  {name:10s} ({label}): {cnt:>6,}  ({pct:.1f}%)  {bar}')

In [ ]:
# Sample reviews from each class
print('=' * 70)
print('SAMPLE REVIEWS FROM EACH CLASS')
print('=' * 70)

for label, name in enumerate(['Negative', 'Neutral', 'Positive']):
    samples = df[df['label'] == label]['text'].sample(2, random_state=42).tolist()
    print(f'\n🔹 {name.upper()}:')
    for i, s in enumerate(samples, 1):
        print(f'  {i}. "{s[:120]}..."' if len(s) > 120 else f'  {i}. "{s}"')

In [ ]:
# Text length statistics
df['word_count']  = df['text'].str.split().str.len()
df['char_count']  = df['text'].str.len()

print('Text Length Statistics (words):')
print(df.groupby('label')['word_count'].describe().round(2).rename(index={0:'Negative',1:'Neutral',2:'Positive'}))

In [ ]:
# Visualize class distribution
DARK_BG  = '#0f172a'
SURFACE  = '#1e293b'
BORDER   = '#334155'
TEXT_C   = '#e2e8f0'
MUTED    = '#94a3b8'
POSITIVE = '#22c55e'
NEGATIVE = '#ef4444'
NEUTRAL  = '#f59e0b'
PRIMARY  = '#6366f1'

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor(DARK_BG)
for ax in axes:
    ax.set_facecolor(SURFACE)
    ax.tick_params(colors=MUTED)
    for spine in ax.spines.values():
        spine.set_edgecolor(BORDER)

# Bar chart
labels  = ['Negative', 'Neutral', 'Positive']
colors  = [NEGATIVE, NEUTRAL, POSITIVE]
counts  = [df['label'].value_counts().sort_index()[i] for i in range(3)]

bars = axes[0].bar(labels, counts, color=colors, alpha=0.85, width=0.5)
for bar in bars:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                 f'{int(bar.get_height()):,}', ha='center', va='bottom', color=TEXT_C, fontsize=11)
axes[0].set_title('Sample Counts', color=TEXT_C, fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count', color=TEXT_C)
axes[0].tick_params(labelcolor=MUTED)
axes[0].grid(axis='y', alpha=0.12, color=BORDER)

# Pie
axes[1].pie(counts, labels=labels, colors=colors, autopct='%1.1f%%',
             textprops=dict(color=TEXT_C), wedgeprops=dict(linewidth=2, edgecolor=DARK_BG))
axes[1].set_title('Proportion', color=TEXT_C, fontsize=13, fontweight='bold')

# Word count distribution per class
for label, name, color in zip([0,1,2], labels, colors):
    sub = df[df['label']==label]['word_count']
    axes[2].hist(sub.clip(upper=200), bins=40, alpha=0.55, color=color, label=name, density=True)
axes[2].set_xlabel('Word Count', color=TEXT_C)
axes[2].set_ylabel('Density', color=TEXT_C)
axes[2].set_title('Review Length Distribution', color=TEXT_C, fontsize=13, fontweight='bold')
axes[2].legend(facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT_C)
axes[2].tick_params(labelcolor=MUTED)
axes[2].grid(alpha=0.1, color=BORDER)

fig.suptitle(f'Amazon Reviews Dataset — {len(df):,} Samples', color=TEXT_C, fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/nb_dataset_overview.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()
print('\n✅ Chart saved: reports/nb_dataset_overview.png')

## 3. Text Preprocessing Demonstration

In [ ]:
from utils.text_cleaner import clean_text

test_cases = [
    "<p>This product is <b>AMAZING</b>!!! Visit http://shop.com for more. Sooooo good!!!</p>",
    "@JohnDoe Terrible service. #disappointed Would NOT recommend.",
    "The item arrived. It's okay... nothing special &amp; not bad either.",
    "café ñoño résumé — unicode test",
    "wtf this broke immediately wtfffff",
]

print('TEXT PREPROCESSING DEMONSTRATION')
print('=' * 70)
for raw in test_cases:
    cleaned = clean_text(raw)
    print(f'\nBefore: {raw}')
    print(f'After : {cleaned}')
    print('-' * 40)

## 4. Train / Val / Test Split Analysis

In [ ]:
from utils.data_loader import split_dataset, compute_class_weights, print_dataset_report

df_train, df_val, df_test = split_dataset(df)

print_dataset_report(df, df_train, df_val, df_test)

# Class weights for imbalance
weights = compute_class_weights(df_train['label'].tolist())
print('\nClass Weights (for CrossEntropyLoss):')
for name, w in zip(['Negative', 'Neutral', 'Positive'], weights):
    print(f'  {name:10s}: {w:.4f}')
print('\nHigher weight = model penalized more for errors on that class')

In [ ]:
from utils.visualizations import plot_dataset_split, plot_class_distribution

os.makedirs('../reports', exist_ok=True)

# Plot the split bar
split_path = plot_dataset_split(len(df_train), len(df_val), len(df_test), '../reports')
img = plt.imread(split_path)
plt.figure(figsize=(12, 4))
plt.imshow(img)
plt.axis('off')
plt.title('Dataset Split Visualization', color='white')
plt.show()
print(f'\nSplit: {len(df_train):,} train | {len(df_val):,} val | {len(df_test):,} test')

In [ ]:
# Verify stratification — class ratios should be ~equal across all splits
print('STRATIFICATION VERIFICATION')
print('(Class proportions should be ~33% each for balanced dataset)')
print('-' * 50)

for split_name, split_df in [('Full', df), ('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    dist = split_df['label'].value_counts(normalize=True).sort_index()
    neg, neu, pos = dist.get(0,0)*100, dist.get(1,0)*100, dist.get(2,0)*100
    print(f'{split_name:6s}: Neg={neg:.1f}%  Neu={neu:.1f}%  Pos={pos:.1f}%  [n={len(split_df):,}]')

## 5. Model Architecture

### Why BERT?

| Feature | TF-IDF + SVM | BiLSTM | **BERT (Ours)** |
|---------|-------------|--------|--------|
| Context | ❌ Bag of words | ✅ Sequential | ✅ Full bidirectional |
| Negation | ❌ | Partial ✅ | ✅ Full |
| Transfer learning | ❌ | ❌ | ✅ 3.3B word pre-training |
| Accuracy (Amazon) | ~78% | ~84% | **~90%** |
| Training time | Minutes | Hours | **~3 epochs** |

### Why VADER Hybrid?
- VADER provides instant lexicon-based signal — no training needed
- Acts as a corrective layer when BERT confidence < 60%
- Improves robustness on out-of-domain text (Twitter, news, formal reports)

### Why Sarcasm Detection?
- "Oh great, it broke immediately" → BERT says Positive → **WRONG**
- Sarcasm flips polarity — must detect and correct

In [ ]:
from models.bert_model import BertSentimentClassifier
import torch.nn as nn

model = BertSentimentClassifier(model_name=cfg.BERT_MODEL_NAME, num_labels=3)

# Count parameters
total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
bert_params     = sum(p.numel() for p in model.bert.parameters())
head_params     = sum(p.numel() for p in model.classifier.parameters())

print('MODEL PARAMETER SUMMARY')
print('=' * 50)
print(f'Total parameters       : {total_params:>12,}')
print(f'Trainable parameters   : {trainable_params:>12,}')
print(f'  BERT backbone        : {bert_params:>12,}  ({bert_params/total_params*100:.1f}%)')
print(f'  Classification head  : {head_params:>12,}  ({head_params/total_params*100:.1f}%)')
print('=' * 50)
print(f'\nClassification head:')
for name, module in model.classifier.named_modules():
    if hasattr(module, 'in_features'):
        print(f'  Linear: {module.in_features} → {module.out_features}')

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained(cfg.BERT_MODEL_NAME)

sample_text = "This product is absolutely fantastic! Best purchase I've made."
enc = tokenizer(sample_text, return_tensors='pt', max_length=256, padding=True, truncation=True)

print('TOKENIZATION DEMO')
print('=' * 60)
print(f'Input text          : "{sample_text}"')
print(f'Tokens              : {tokenizer.convert_ids_to_tokens(enc["input_ids"][0].tolist())}')
print(f'Token count         : {enc["input_ids"].shape[1]}')
print(f'[CLS] token id      : {enc["input_ids"][0][0].item()}  → used for classification')

# Forward pass
model.eval()
with torch.no_grad():
    logits = model(**enc)
probs = torch.softmax(logits, dim=-1)[0]

print(f'\nModel output (untrained, random):')
for label, prob in zip(['Negative', 'Neutral', 'Positive'], probs.tolist()):
    bar = '█' * int(prob * 40)
    print(f'  {label:10s}: {prob:.4f}  {bar}')
print('(Note: this is BEFORE fine-tuning — use trained checkpoint for real predictions)')

## 6. Training Loop Demo (Mini)

This cell runs **1 mini epoch** (50 batches) to demonstrate the training loop structure.
For full training, use `python train.py` from the terminal.

In [ ]:
from models.bert_model import BertSentimentTrainer, SentimentDataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score

MINI_DEMO = True   # Set False to run full epoch

# Use a tiny subset for notebook demo
mini_train = df_train.head(500).reset_index(drop=True)
mini_val   = df_val.head(100).reset_index(drop=True)

trainer = BertSentimentTrainer(cfg=cfg)

mini_loader_train = DataLoader(
    SentimentDataset(mini_train['text'].tolist(), mini_train['label'].tolist(), trainer.tokenizer, 128),
    batch_size=8, shuffle=True
)
mini_loader_val = DataLoader(
    SentimentDataset(mini_val['text'].tolist(), mini_val['label'].tolist(), trainer.tokenizer, 128),
    batch_size=16, shuffle=False
)

optimizer = torch.optim.AdamW(trainer.model.parameters(), lr=2e-5)
criterion = torch.nn.CrossEntropyLoss()

print('Running 1 mini epoch (500 samples, batch=8) for demonstration...')
print('This shows the training loop. For real training, use: python train.py')
print('=' * 60)

trainer.model.train()
batch_losses = []
for i, batch in enumerate(mini_loader_train):
    optimizer.zero_grad()
    logits = trainer.model(
        batch['input_ids'].to(trainer.device),
        batch['attention_mask'].to(trainer.device),
        batch['token_type_ids'].to(trainer.device),
    )
    loss = criterion(logits, batch['label'].to(trainer.device))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(trainer.model.parameters(), 1.0)
    optimizer.step()
    batch_losses.append(loss.item())
    
    if (i+1) % 10 == 0:
        print(f'  Batch {i+1:3d}/{len(mini_loader_train)} | Loss: {loss.item():.4f} | Avg: {np.mean(batch_losses):.4f}')
    
    if MINI_DEMO and i >= 29:
        print('  [Demo mode: stopping after 30 batches]')
        break

print(f'\nFinal avg loss: {np.mean(batch_losses):.4f}')
print('\n⚠️  This is an untrained model demo. Run python train.py for proper training.')

## 7. Evaluation Metrics Demo

Load the **trained checkpoint** (if available) and evaluate on the test set.

In [ ]:
from utils.metrics import compute_metrics, print_metrics
import json

CHECKPOINT = '../data/checkpoints/bert_sentiment'
trained_available = os.path.exists(CHECKPOINT)

if trained_available:
    print(f'✅ Loading trained checkpoint: {CHECKPOINT}')
    eval_trainer = BertSentimentTrainer(cfg=cfg)
    eval_trainer.load(CHECKPOINT)
    
    print(f'Running evaluation on {len(df_test):,} test samples...')
    preds, confs, probs = eval_trainer.predict_batch(df_test['text'].tolist())
    true_labels = df_test['label'].tolist()
    
    metrics = compute_metrics(true_labels, preds, label_names=cfg.SENTIMENT_LABELS)
    print_metrics(metrics)
else:
    print('⚠️  No trained checkpoint found.')
    print(f'   Expected at: {CHECKPOINT}')
    print('   Run: python train.py --max-samples 20000 --epochs 2')
    print()
    print('   Demonstrating with VADER-only predictions instead...')
    
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    vader = SentimentIntensityAnalyzer()
    
    def vader_predict(text):
        c = vader.polarity_scores(text)['compound']
        return 0 if c <= -0.05 else (2 if c >= 0.05 else 1)
    
    sample = df_test.head(200)
    preds  = sample['text'].apply(vader_predict).tolist()
    true_labels = sample['label'].tolist()
    metrics = compute_metrics(true_labels, preds, label_names=cfg.SENTIMENT_LABELS)
    print('\n📊 VADER-only metrics (200 test samples):')
    print_metrics(metrics)

## 8. Visualizations

Load saved charts or generate from training history.

In [ ]:
import os

report_dir = '../reports'
chart_files = [
    ('training_dashboard.png', 'All-in-One Training Dashboard'),
    ('loss_curve.png',          'Training vs Validation Loss'),
    ('accuracy_curve.png',      'Validation Accuracy & F1'),
    ('confusion_matrix.png',    'Confusion Matrix (Test Set)'),
    ('per_class_metrics.png',   'Per-Class Metrics (Final Epoch)'),
    ('dataset_split.png',       'Train/Val/Test Split'),
]

available = [(f, t) for f, t in chart_files if os.path.exists(os.path.join(report_dir, f))]

if not available:
    print('No charts found yet. Run python train.py to generate them.')
    print('\nGenerating confusion matrix from current results...')
    from utils.visualizations import plot_confusion_matrix
    os.makedirs(report_dir, exist_ok=True)
    path = plot_confusion_matrix(true_labels, preds, report_dir, 'Confusion Matrix (Demo)')
    available = [(os.path.basename(path), 'Confusion Matrix (Demo)')]

for fname, title in available:
    fpath = os.path.join(report_dir, fname)
    if os.path.exists(fpath):
        img = plt.imread(fpath)
        h, w = img.shape[:2]
        fig, ax = plt.subplots(figsize=(14, 14*h/w))
        fig.patch.set_facecolor('#0f172a')
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(title, color='white', fontsize=13, fontweight='bold', pad=10)
        plt.tight_layout()
        plt.show()
        print(f'📊 {title}: {fpath}')

In [ ]:
# If training_log.csv exists, plot directly in notebook
log_path = os.path.join(report_dir, 'training_log.csv')

if os.path.exists(log_path):
    log_df = pd.read_csv(log_path)
    print('Training Log:')
    display(log_df.round(4))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor(DARK_BG)
    for ax in axes:
        ax.set_facecolor(SURFACE)
        ax.tick_params(colors=MUTED)
        for sp in ax.spines.values(): sp.set_edgecolor(BORDER)
    
    axes[0].plot(log_df['epoch'], log_df['train_loss'], color=PRIMARY, marker='o', lw=2.5, ms=7, label='Train Loss')
    axes[0].plot(log_df['epoch'], log_df['val_loss'],   color='#38bdf8', marker='s', lw=2.5, ms=7, label='Val Loss', ls='--')
    axes[0].set_title('Loss Curves', color=TEXT_C, fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch', color=TEXT_C)
    axes[0].set_ylabel('Loss', color=TEXT_C)
    axes[0].legend(facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT_C)
    axes[0].grid(alpha=0.12, color=BORDER)
    axes[0].fill_between(log_df['epoch'], log_df['train_loss'], alpha=0.12, color=PRIMARY)
    axes[0].fill_between(log_df['epoch'], log_df['val_loss'],   alpha=0.12, color='#38bdf8')
    
    axes[1].plot(log_df['epoch'], log_df['val_accuracy'], color=POSITIVE, marker='o', lw=2.5, ms=7, label='Accuracy')
    axes[1].plot(log_df['epoch'], log_df['val_f1'],       color=NEUTRAL,  marker='D', lw=2.5, ms=7, label='Macro F1', ls='--')
    axes[1].plot(log_df['epoch'], log_df['val_f1_neg'],   color=NEGATIVE, marker='^', lw=1.5, ms=5, label='F1 Neg', ls=':')
    axes[1].plot(log_df['epoch'], log_df['val_f1_pos'],   color=POSITIVE, marker='v', lw=1.5, ms=5, label='F1 Pos', ls=':')
    axes[1].set_ylim(0, 1.05)
    axes[1].set_title('Validation Metrics per Epoch', color=TEXT_C, fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch', color=TEXT_C)
    axes[1].set_ylabel('Score', color=TEXT_C)
    axes[1].legend(facecolor=SURFACE, edgecolor=BORDER, labelcolor=TEXT_C)
    axes[1].grid(alpha=0.12, color=BORDER)
    
    plt.suptitle('Training Progress', color=TEXT_C, fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(report_dir, 'nb_training_curves.png'), dpi=150, bbox_inches='tight', facecolor=DARK_BG)
    plt.show()
else:
    print(f'No training log found at {log_path}')
    print('Run: python train.py to generate training data')

## 9. Live Prediction Demo

Demonstrates the full hybrid BERT + VADER pipeline.

In [ ]:
from services.sentiment_service import SentimentService

service = SentimentService(cfg=cfg)
service.load_models()   # loads BERT if checkpoint exists, else VADER-only

test_reviews = [
    ("This product is absolutely amazing! Best purchase ever. Highly recommend!", "Expected: Positive"),
    ("Terrible quality. Broke after one day. Complete waste of money. Do not buy.", "Expected: Negative"),
    ("The item arrived on time and works as described. Nothing special.", "Expected: Neutral"),
    ("Oh great, another product that breaks immediately. Just what I needed.", "Expected: Negative (sarcasm)"),
    ("Battery life is great but the screen quality is disappointing.", "Expected: Neutral (mixed)"),
    ("meh", "Expected: Neutral (short text)"),
]

print('LIVE PREDICTION DEMO — Hybrid BERT + VADER + Sarcasm Pipeline')
print('=' * 70)

icons = {'Positive': '✅', 'Negative': '❌', 'Neutral': '⚪'}

for review, expected in test_reviews:
    r = service.predict(review)
    icon = icons.get(r['predicted_sentiment'], '?')
    sarc = '😏 SARCASM' if r['sarcasm_detected'] else ''
    
    print(f'\nReview   : "{review[:80]}"')
    print(f'{expected}')
    print(f'Result   : {icon} {r["predicted_sentiment"]:10s}  (conf={r["confidence_score"]:.2%}) {sarc}')
    print(f'BERT     : {r["bert_sentiment"]:10s}  VADER: {r["vader_sentiment"]:10s}  compound={r["vader_scores"]["compound"]:.3f}')
    p = r['probabilities']
    print(f'Probs    : Neg={p["negative"]:.3f}  Neu={p["neutral"]:.3f}  Pos={p["positive"]:.3f}')
    print(f'Time     : {r["processing_time_ms"]:.1f}ms')
    print('-' * 60)

## 10. BERT vs VADER Comparison

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

tricky_cases = [
    "This is the worst best product I've ever not hated.",         # Confusing negation
    "Not bad at all!",                                              # Double negation
    "The price is a bit high but the quality makes up for it.",   # Mixed
    "If you like mediocre products, this is perfect for you.",    # Sarcasm
    "Works. Received it. Used it.",                               # Neutral/factual
    "ABSOLUTELY LOVE THIS!!! 10/10 would recommend 🔥🔥",         # Caps + emoji
]

print('BERT vs VADER COMPARISON ON TRICKY CASES')
print('=' * 70)
print(f'{"Review":<45} {"VADER":>10} {"Hybrid":>12} {"Compound":>10}')
print('-' * 70)

for text in tricky_cases:
    vader_scores = vader.polarity_scores(text)
    c = vader_scores['compound']
    vader_lbl = 'Negative' if c <= -0.05 else ('Positive' if c >= 0.05 else 'Neutral')
    
    r = service.predict(text)
    hybrid_lbl = r['predicted_sentiment']
    
    display_text = text[:44] + ('...' if len(text) > 44 else '')
    match = '✓' if vader_lbl == hybrid_lbl else '≠'
    print(f'{display_text:<45} {vader_lbl:>10} {hybrid_lbl:>12}  {c:>+.3f}  {match}')

## 11. Sarcasm Detection Demo

In [ ]:
from models.sarcasm_model import SarcasmDetector

detector = SarcasmDetector()
detector.load()   # loads model or falls back to rules

sarcasm_examples = [
    ("Oh great, it broke after one day. Just brilliant.",          True,  "Obvious sarcasm"),
    ("Yeah right, like this will ever work properly.",             True,  "Lexical sarcasm signal"),
    ("Absolutely WONDERFUL that it stopped working within a week.",True,  "Capitalization + sarcasm"),
    ("This is genuinely a great product. I love it.",              False, "Genuine positive"),
    ("The product works as described. Decent quality.",            False, "Neutral factual"),
    ("I can't believe how terrible this is. Worst purchase ever.", False, "Genuine negative (no sarcasm)"),
]

print('SARCASM DETECTION DEMO')
print('=' * 70)
print(f'{"Text":<52} {"True":>6} {"Pred":>6} {"Conf":>7} {"Correct":>7}')
print('-' * 70)

correct = 0
for text, true_sarc, desc in sarcasm_examples:
    is_sarcastic, conf = detector.detect(text)
    match = '✓' if is_sarcastic == true_sarc else '✗'
    correct += int(is_sarcastic == true_sarc)
    display_text = text[:51] + ('...' if len(text)>51 else '')
    print(f'{display_text:<52} {str(true_sarc):>6} {str(is_sarcastic):>6} {conf:>7.3f} {match:>7}')
    print(f'  → [{desc}]')

print(f'\nRule-based accuracy: {correct}/{len(sarcasm_examples)} ({correct/len(sarcasm_examples)*100:.0f}%)')
print('(Train the DistilBERT sarcasm model for higher accuracy)')

## 12. Export Metrics CSV

In [ ]:
# Export current metrics to CSV
export_df = pd.DataFrame([{
    'Model':      'BERT + VADER Hybrid',
    'Dataset':    'Amazon Reviews (amazon_polarity + sst2)',
    'Samples':    len(df),
    'Train':      len(df_train),
    'Val':        len(df_val),
    'Test':       len(df_test),
    'Accuracy':   round(metrics.get('accuracy', 0), 4),
    'Macro_F1':   round(metrics.get('macro_f1', 0), 4),
    'Macro_P':    round(metrics.get('macro_precision', 0), 4),
    'Macro_R':    round(metrics.get('macro_recall', 0), 4),
    'F1_Negative': round(metrics.get('per_class', {}).get('Negative', {}).get('f1', 0), 4),
    'F1_Neutral':  round(metrics.get('per_class', {}).get('Neutral',  {}).get('f1', 0), 4),
    'F1_Positive': round(metrics.get('per_class', {}).get('Positive', {}).get('f1', 0), 4),
}])

os.makedirs('../reports', exist_ok=True)
export_path = '../reports/nb_metrics_export.csv'
export_df.to_csv(export_path, index=False)

print('EXPORTED METRICS:')
print('=' * 60)
for col, val in export_df.iloc[0].items():
    print(f'  {col:20s}: {val}')

print(f'\n✅ Saved to: {export_path}')

In [ ]:
# Show all_metrics.csv if it exists (generated by train.py)
metrics_csv = '../reports/all_metrics.csv'
if os.path.exists(metrics_csv):
    df_metrics = pd.read_csv(metrics_csv)
    print('ALL METRICS (from reports/all_metrics.csv):')
    display(df_metrics.round(4))
else:
    print(f'Run python train.py to generate {metrics_csv}')

## Summary

| Component | Status |
|-----------|--------|
| Dataset loading (Amazon Reviews) | ✅ Implemented |
| Text preprocessing | ✅ Implemented |
| Stratified train/val/test split | ✅ Implemented |
| BERT fine-tuning | ✅ Implemented (run train.py) |
| Per-epoch metrics (loss, acc, P, R, F1) | ✅ Implemented |
| VADER hybrid layer | ✅ Implemented |
| Sarcasm detection | ✅ Implemented |
| Visualizations (7 chart types) | ✅ Implemented |
| Metrics CSV export | ✅ Implemented |
| Flask web API | ✅ Implemented |

---

**To run full training:**
```bash
python train.py --max-samples 60000 --epochs 3
```

**To start the web app:**
```bash
python app.py
# Open: http://localhost:5000
```